# Cài đặt các Thư viện sử dụng

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import joblib
from underthesea import word_tokenize
import os

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Cài đặt hiển thị dataFrame

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Đọc dữ liệu


- vn_news_223_tdlfr.csv là file chứa news cần xử lý
- vietnamese-stopwords.txt chứa các stop word cần xử lý

In [ ]:
file = os.path.join("../../Data_Train", "English.csv")
data_df = pd.read_csv(file, encoding = 'utf-8')

# Khám phá dữ liệu

## Mẫu dữ liệu

In [ ]:
data_df.sample(1)

## Xem thông tin

In [ ]:
data_df.info()

## Xem mô tả

In [ ]:
data_df.describe().round(1)

## Xem số dòng và số cột của dữ liệu

In [ ]:
num_rows=data_df.shape[0]
num_cols=data_df.shape[1]
print(num_rows)
print(num_cols)

## ý nghĩa của các dòng

### Kiểm tra dữ liệu có bị lặp ?

In [ ]:
dup=data_df.index.duplicated().sum()
dup

### ý nghĩa các cột

- text: nội dung của tin tức
- domain: đường dẫn đến trang web chứa tin tức 
- label: nhãn phân biệt tin giả hay tin thật

### Dữ liệu có bị thiếu không?

In [ ]:
data_df['title'].isnull().sum()

In [ ]:
data_df['text'].isnull().sum()

In [ ]:
data_df['label'].isnull().sum()

### Mỗi cột hiện đang có kiểu dữ liệu gì? Có cột nào có kiểu dữ liệu chưa phù hợp để có thể xử lý tiếp không?

In [ ]:
data_df.dtypes

#### Cột có dtype là object nghĩa là sao?

- Trong Pandas, kiểu dữ liệu object thường ám chỉ chuỗi, nhưng thật ra kiểu dữ liệu object có thể chứa một đối tượng bất kỳ trong Python (vì thật ra ở bên dưới kiểu dữ liệu object chứa địa chỉ).
- Nếu một cột trong dataframe có dtype là object thì có thể các phần tử trong cột này sẽ có kiểu dữ liệu khác nhau
- Để biết được kiểu dữ liệu thật sự của các phần tử trong cột này thì ta phải truy xuất vào từng phần tử. Ta muốn xem thử trong nội bộ mỗi cột này có các kiểu dữ liệu nào

In [ ]:
def open_object_dtype(s):
    dtypes = set()
    s=s.apply(type)
    dtypes.update(s.unique().tolist())
    return dtypes

In [ ]:
open_object_dtype(data_df['title'])

In [ ]:
open_object_dtype(data_df['text'])

In [ ]:
open_object_dtype(data_df['label'])

In [ ]:
data_df.isnull().sum()

#### Kiểm tra phân bố các class có chênh lệch không?

Xem phân bố dữ liệu

In [ ]:
# data quantity chart
def quantity_chart():
    counts_df1 = data_df['label'].value_counts()

    plt.figure(figsize=(6, 6))

    plt.pie(counts_df1, labels=counts_df1, autopct='%1.1f%%', startangle=90)
    plt.title('Bảng phân phối tin thật và tin giả')

    labels = ['Real' if label == 0 else 'Fake' for label in counts_df1.index]
    plt.legend(labels=labels, loc="best", fontsize=18)  # Tăng kích thước nhãn trong chú thích
    plt.tight_layout()
    plt.show()
quantity_chart()

#### Các thông tin thống kê

##### Chiều dài trung bình mỗi record là bao nhiêu?

In [ ]:
len_sum=0
for i in data_df['text']:
    len_sum+=len(i)
len_avg=len_sum/data_df['text'].count()
len_avg

##### Record dài nhất là bao nhiêu?

In [ ]:
max_len=0
for i in data_df['text']:
    if len(i)>max_len:
        max_len=len(i)
max_len

##### Record ngắn nhất là bao nhiêu?

In [ ]:
min_len=len(data_df['text'][0])
for i in data_df['text']:
    if len(i)<min_len:
        min_len=len(i)
min_len

### Tiền xử lí văn bản

#### Loại bỏ các đường link và các dấu câu, lowercase

In [ ]:
import nltk
from nltk.corpus import stopwords

# Tải stopwords từ NLTK nếu chưa tải
# nltk.download('stopwords')

# Sử dụng danh sách từ dừng (stopwords) của tiếng Anh từ NLTK
stopword_list = list(stopwords.words('english'))
stopwords = [word for word in stopword_list if word.strip() != '']


In [ ]:
import string
def wordopt(text):
    text=re.sub("Reuters"," ",text)

    update_text =""

    text = text.lower()

    text=re.sub("reuters"," ",text)
    
    #simplifying text
    text=re.sub(r"i'm","i am",text)
    text=re.sub(r"he's","he is",text)
    text=re.sub(r"she's","she is",text)
    text=re.sub(r"that's","that is",text)
    text=re.sub(r"what's","what is",text)
    text=re.sub(r"where's","where is",text)
    text=re.sub(r"\'ll"," will",text)
    text=re.sub(r"\'ve"," have",text)
    text=re.sub(r"\'re"," are",text)
    text=re.sub(r"\'d"," would",text)
    text=re.sub(r"won't","will not",text)
    text=re.sub(r"can't","cannot",text)


    text=re.sub(r"[-()\"#!@$%^&*{}?.,:]"," ",text)
    text=re.sub(r"\s+"," ",text)

    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W", " ", text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)

    for word in text.split():
        if word not in stopwords:
            update_text += word+" "
    
    return update_text.strip()


In [ ]:
data_df['article'] = ' |title| '+ data_df["title"] +' |text| '+  data_df["text"] 

In [ ]:
data_df.head(1)

In [ ]:
data_df['Compound_Content_SW'] = data_df['article'].apply(wordopt)

In [ ]:
data_df.head(1)

# Vector hóa

#### Tokenizer

In [ ]:
def tokenize(sentence):
    return word_tokenize(sentence, format = 'word')

### Mô hình hóa

- Chuyển đoạn văn tiếng Việt về vector, sử dụng CountVectorizer của sklearn
- với các tham số là danh sách stopwords tiếng Việt
- và tokenizer tách từ tiếng Việt

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

Chia tập dữ liệu thành 2 tập

- Tập train: 75%
- Tập test: 25%


In [ ]:
X = data_df["Compound_Content_SW"]
y = data_df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.25,
                                                   random_state=30)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB,GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


## CountVectorizer

In [ ]:
model_vector_CV = os.path.join("../../Model/English/CV", "English_vectorizer_CV.joblib")
model_DTC_CV = os.path.join("../../Model/English/CV", "English_DTC_model_CV.joblib")
model_NB_CV = os.path.join("../../Model/English/CV", "English_NB_model_CV.joblib")
model_RFC_CV = os.path.join("../../Model/English/CV", "English_RFC_model_CV.joblib")
model_SVM_CV = os.path.join("../../Model/English/CV", "English_SVM_model_CV.joblib")


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
vectorizerCV = CountVectorizer(
    stop_words = stopwords,
    tokenizer = tokenize,
)

In [ ]:
Xv_trainCV = vectorizerCV.fit_transform(X_train)
Xv_testCV = vectorizerCV.transform(X_test)

In [ ]:
joblib.dump(vectorizerCV, model_vector_CV)

### DecisionTreeClassifier

In [ ]:
dtc_cv = DecisionTreeClassifier()
dtc_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_dt_cv = dtc_cv.predict(Xv_testCV)
print(dtc_cv.score(Xv_testCV, y_test))
print(classification_report(y_true=y_test, y_pred=pred_dt_cv))

In [ ]:
joblib.dump(dtc_cv, model_DTC_CV)

### Naive Bayes

In [ ]:
nb_cv = MultinomialNB()
nb_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_np_cv = nb_cv.predict(Xv_testCV)
print(nb_cv.score(Xv_testCV, y_test))
print(classification_report(y_true=y_test, y_pred=pred_np_cv))

In [ ]:
joblib.dump(nb_cv, model_NB_CV)

### RandomForest

In [ ]:
rfc_cv = RandomForestClassifier()
rfc_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_rfc_cv = rfc_cv.predict(Xv_testCV)
print(rfc_cv.score(Xv_testCV, y_test))
print(classification_report(y_true=y_test, y_pred=pred_rfc_cv))

In [ ]:
joblib.dump(rfc_cv, model_RFC_CV)

### SVM

In [ ]:
svm_cv = SVC(kernel='linear')
svm_cv.fit(Xv_trainCV, y_train)

In [ ]:
pred_svm_cv = svm_cv.predict(Xv_testCV)

accuracy = accuracy_score(y_test, pred_svm_cv)  # Tính độ chính xác
print(accuracy)
print("Accuracy:", svm_cv.score(Xv_testCV, y_test))  # Đánh giá mô hình với nhãn thực tế

print(classification_report(y_true=y_test, y_pred=pred_svm_cv))

In [ ]:
joblib.dump(svm_cv, model_SVM_CV)

## TfidfVectorizer

In [ ]:
model_vector_TF = os.path.join("../../Model/English/TF", "English_vectorizer_TF.joblib")
model_DTC_TF = os.path.join("../../Model/English/TF", "English_DTC_model_TF.joblib")
model_NB_TF = os.path.join("../../Model/English/TF", "English_NB_model_TF.joblib")
model_RFC_TF = os.path.join("../../Model/English/TF", "English_RFC_model_TF.joblib")
model_SVM_TF = os.path.join("../../Model/English/TF", "English_SVM_model_TF.joblib")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizerTF = TfidfVectorizer(stop_words=stopwords)

In [ ]:
Xv_trainTF = vectorizerTF.fit_transform(X_train)
Xv_testTF = vectorizerTF.transform(X_test)

In [ ]:
joblib.dump(vectorizerTF, model_vector_TF)

### DecisionTreeClassifier

In [ ]:
dtc_tf = DecisionTreeClassifier()
dtc_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_dtc_tf = dtc_tf.predict(Xv_testTF)
print(dtc_tf.score(Xv_testTF, y_test))
print(classification_report(y_true=y_test, y_pred=pred_dtc_tf))

In [ ]:
joblib.dump(dtc_tf, model_DTC_TF)

### Navie bayes

In [ ]:
nb_tf = MultinomialNB()
nb_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_nb_tf = nb_tf.predict(Xv_testTF)
print(nb_tf.score(Xv_testTF, y_test))
print(classification_report(y_true=y_test, y_pred=pred_nb_tf))

In [ ]:
#save model Naive Bayes
joblib.dump(nb_tf, model_NB_TF)

### RandomForest

In [ ]:
rfc_tf = RandomForestClassifier()
rfc_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_rfc_tf = rfc_tf.predict(Xv_testTF)
print(rfc_tf.score(Xv_testTF, y_test))
print(classification_report(y_true=y_test, y_pred=pred_rfc_tf))

In [ ]:
joblib.dump(rfc_tf, model_RFC_TF)

### SVM

In [ ]:
svm_tf = SVC(kernel='linear')
svm_tf.fit(Xv_trainTF, y_train)

In [ ]:
pred_svm_tf = svm_tf.predict(Xv_testTF)

accuracy = accuracy_score(y_test, pred_svm_tf)  # Tính độ chính xác
print(accuracy)
print("Accuracy:", svm_tf.score(Xv_testTF, y_test))  # Đánh giá mô hình với nhãn thực tế

print(classification_report(y_true=y_test, y_pred=pred_svm_tf))

In [ ]:
joblib.dump(svm_tf, model_SVM_TF)

## Word2Vec

In [ ]:
model_vector_W2V = os.path.join("../../Model/English/W2V", "English_vectorizer_W2V.joblib")
model_DTC_W2V = os.path.join("../../Model/English/W2V", "English_DTC_model_W2V.joblib")
model_NB_W2V = os.path.join("../../Model/English/W2V", "English_NB_model_W2V.joblib")
model_RFC_W2V = os.path.join("../../Model/English/W2V", "English_RFC_model_W2V.joblib")
model_SVM_W2V = os.path.join("../../Model/English/W2V", "English_SVM_model_W2V.joblib")

In [ ]:
from gensim.models import Word2Vec
import numpy as np

In [ ]:
def makeWords(sentences):
    wordList = []
    for sentence in sentences:
        words = sentence.split(' ')
        wordList.append(words)
    return wordList

words_train_W2V = makeWords(X_train)
words_test_W2V = makeWords(X_test)

In [ ]:
# Train the Word2Vec model on the tokenized training sentences
vectorizerW2V = Word2Vec(
    sentences=words_train_W2V,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0,
    epochs=10
)

In [ ]:
# Function to calculate the average Word2Vec vector for each sentence
def sentence_vector(sentence, model):
    # Get vectors for words in the sentence if they exist in the vocabulary
    vectors = [model.wv[word] for word in sentence if word in model.wv]
    if len(vectors) > 0:
        # Calculate the mean of the word vectors for the sentence
        return np.mean(vectors, axis=0)
    else:
        # Return a zero vector if no words in the sentence are in the vocabulary
        return np.zeros(model.vector_size)

In [ ]:
Xv_trainW2V = np.array([sentence_vector(sentence, vectorizerW2V) for sentence in words_train_W2V])
Xv_testW2V = np.array([sentence_vector(sentence, vectorizerW2V) for sentence in words_test_W2V])

In [ ]:
joblib.dump(vectorizerW2V,model_vector_W2V)

### Decision Tree

In [ ]:
dtc_w2v = DecisionTreeClassifier()
dtc_w2v.fit(Xv_trainW2V, y_train)

In [ ]:
pred_dtc_w2v = dtc_w2v.predict(Xv_testW2V)
print(dtc_w2v.score(Xv_testW2V, y_test))
accuracy = accuracy_score(y_test, pred_dtc_w2v) 
print(accuracy)
print(classification_report(y_true=y_test, y_pred=pred_dtc_w2v))

In [ ]:
# save model Decision Tree
joblib.dump(dtc_w2v, model_DTC_W2V)

### Navie bayes

In [ ]:
nb_w2v = GaussianNB()
nb_w2v.fit(Xv_trainW2V,y_train)

In [ ]:
pred_np_w2v = nb_w2v.predict(Xv_testW2V)
print(nb_w2v.score(Xv_testW2V, y_test))
print(classification_report(y_true=y_test, y_pred=pred_np_w2v))

In [ ]:
joblib.dump(nb_w2v, model_NB_W2V)

### RandomForest

In [ ]:
rfc_w2v = RandomForestClassifier()
rfc_w2v.fit(Xv_trainW2V, y_train)

In [ ]:
pred_rfc_w2v = rfc_w2v.predict(Xv_testW2V)
print(rfc_w2v.score(Xv_testW2V, y_test))
accuracy = accuracy_score(y_test, pred_rfc_w2v) 
print(accuracy)
print(classification_report(y_true=y_test, y_pred=pred_rfc_w2v))

In [ ]:
joblib.dump(rfc_w2v, model_RFC_W2V)

### SVM

In [ ]:
svm_w2v = SVC(kernel='linear')
svm_w2v.fit(Xv_trainW2V, y_train)

In [ ]:
pred_svm_w2v = svm_w2v.predict(Xv_testW2V)

# Đánh giá mô hình
# print("Accuracy:", svm_model.score(X_test_vec, y_pred))
accuracy = accuracy_score(y_test, pred_svm_w2v) 
print(accuracy)
print("Accuracy:", svm_w2v.score(Xv_testW2V, y_test))  # Đánh giá mô hình với nhãn thực tế

print(classification_report(y_true=y_test, y_pred=pred_svm_w2v))

In [ ]:
joblib.dump(svm_w2v, model_SVM_W2V)

## Doc2Vec

In [ ]:
model_vector_D2V = os.path.join("../../Model/English/D2V", "English_vectorizer_D2V.joblib")
model_DTC_D2V = os.path.join("../../Model/English/D2V", "English_DTC_model_D2V.joblib")
model_NB_D2V = os.path.join("../../Model/English/D2V", "English_NB_model_D2V.joblib")
model_RFC_D2V = os.path.join("../../Model/English/D2V", "English_RFC_model_D2V.joblib")
model_SVM_D2V = os.path.join("../../Model/English/D2V", "English_SVM_model_D2V.joblib")

In [ ]:
from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

In [ ]:
documents = [TaggedDocument(doc.split(), [i]) for i, doc in enumerate(X_train)]

In [ ]:
vectorizerD2V = Doc2Vec(vector_size=50, min_count=1, epochs=40)
vectorizerD2V.build_vocab(documents)
vectorizerD2V.train(documents, total_examples=vectorizerD2V.corpus_count, epochs=vectorizerD2V.epochs)

In [ ]:
joblib.dump(vectorizerD2V, model_vector_D2V)

In [ ]:
Xv_trainD2V = [vectorizerD2V.infer_vector(doc.split()) for doc in X_train]
Xv_testD2V = [vectorizerD2V.infer_vector(doc.split()) for doc in X_test]

In [ ]:
joblib.dump(vectorizerD2V, model_vector_D2V)

### DecisionTreeClassifier

In [ ]:
dtc_d2v = DecisionTreeClassifier()
dtc_d2v.fit(Xv_trainD2V, y_train)

In [ ]:
pred_dtc_d2v = dtc_d2v.predict(Xv_testD2V)
print(dtc_d2v.score(Xv_testD2V, y_test))
accuracy = accuracy_score(y_test, pred_dtc_d2v) 
print(accuracy)
print(classification_report(y_true=y_test, y_pred=pred_dtc_d2v))

In [ ]:
joblib.dump(dtc_d2v, model_DTC_D2V)

### Navie bayes

In [ ]:
nb_d2v = GaussianNB()
nb_d2v.fit(Xv_trainD2V,y_train)

In [ ]:
pred_np_d2v = nb_d2v.predict(Xv_testD2V)
print(nb_d2v.score(Xv_testD2V, y_test))
print(classification_report(y_true=y_test, y_pred=pred_np_d2v))

In [ ]:
joblib.dump(nb_d2v, model_NB_D2V)

### RandomForest

In [ ]:
rfc_d2v = RandomForestClassifier()
rfc_d2v.fit(Xv_trainD2V, y_train)

In [ ]:
pred_rfc_d2v = rfc_d2v.predict(Xv_testD2V)
print(rfc_d2v.score(Xv_testD2V, y_test))
accuracy = accuracy_score(y_test, pred_rfc_d2v) 
print(accuracy)
print(classification_report(y_true=y_test, y_pred=pred_rfc_d2v))

In [ ]:
joblib.dump(rfc_d2v, model_RFC_D2V)

### SVM

In [ ]:
svm_d2v = SVC(kernel='linear')
svm_d2v.fit(Xv_trainD2V, y_train)

In [ ]:
pred_svm_d2v = svm_d2v.predict(Xv_testD2V)

# Đánh giá mô hình
# print("Accuracy:", svm_model.score(X_test_vec, y_pred))
accuracy = accuracy_score(y_test, pred_svm_d2v) 
print(accuracy)
print("Accuracy:", svm_d2v.score(Xv_testD2V, y_test))  # Đánh giá mô hình với nhãn thực tế

print(classification_report(y_true=y_test, y_pred=pred_svm_d2v))

In [ ]:
joblib.dump(svm_d2v, model_SVM_D2V)